# Fashion-MNIST con redes densas (Flatten)

Tarea: entrenar una red neuronal con `Flatten`, probar **3 configuraciones** distintas y comparar usando **accuracy**.

Dataset: **Fashion-MNIST** (70,000 imágenes 28×28 en escala de grises, 10 clases de ropa).

In [ ]:
import numpy as np
import tensorflow as tf
import tensorflow_datasets as tfds
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

tf.random.set_seed(42)
np.random.seed(42)

gpus = tf.config.list_physical_devices('GPU')
print("GPUs:", gpus)

In [ ]:
def plot_learning_curve(history, titulo=""):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history.history['accuracy'], label='Training accuracy')
    axes[0].plot(history.history['val_accuracy'], label='Validation accuracy')
    axes[0].set_title(f'Accuracy - {titulo}')
    axes[0].set_xlabel('Época')
    axes[0].legend()

    axes[1].plot(history.history['loss'], label='Training loss')
    axes[1].plot(history.history['val_loss'], label='Validation loss')
    axes[1].set_title(f'Loss - {titulo}')
    axes[1].set_xlabel('Época')
    axes[1].legend()
    plt.tight_layout()
    plt.show()

## 1. Carga de datos

In [ ]:
train_ds, test_ds = tfds.load(
    'fashion_mnist',
    split=['train', 'test'],
    shuffle_files=True,
    as_supervised=True,
    batch_size=32,
)

class_names = ["Camiseta", "Pantalón", "Suéter", "Vestido", "Abrigo",
               "Sandalia", "Camisa", "Tenis", "Bolsa", "Botín"]

In [ ]:
# Visualizar imágenes de ejemplo (cada imagen es una matriz 2D de 28x28)
imgs, labels = next(iter(train_ds))
print("Forma de un batch:", imgs.shape)   # (32, 28, 28, 1)

plt.figure(figsize=(12, 5))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(imgs[i].numpy().squeeze(), cmap="gray")
    plt.title(class_names[labels[i]])
    plt.axis("off")
plt.tight_layout()
plt.show()

## 2. Preprocesamiento
Normalizamos los pixeles a [0, 1].

In [ ]:
def normalizar(x, y):
    return tf.cast(x, tf.float32) / 255.0, y

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.map(normalizar, num_parallel_calls=AUTOTUNE).cache().prefetch(AUTOTUNE)
test_ds  = test_ds.map(normalizar,  num_parallel_calls=AUTOTUNE).cache().prefetch(AUTOTUNE)

INPUT_SHAPE = (28, 28, 1)
EPOCHS = 15

## 3. Configuraciones

| Config | Arquitectura | Regularización | Optimizador |
|---|---|---|---|
| A | Flatten → 128 → 10 | — | Adam (lr=1e-3) |
| B | Flatten → 256 → 128 → 10 | Dropout 0.3 | Adam (lr=1e-3) |
| C | Flatten → 512 → 256 → 128 → 10 | BatchNorm + Dropout 0.3 | SGD + momentum (lr=1e-2) |

In [ ]:
def config_a():
    return tf.keras.Sequential([
        tf.keras.layers.Input(shape=INPUT_SHAPE),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(128, activation="relu"),
        tf.keras.layers.Dense(10, activation="softmax"),
    ], name="A_simple"), tf.keras.optimizers.Adam(1e-3)

def config_b():
    return tf.keras.Sequential([
        tf.keras.layers.Input(shape=INPUT_SHAPE),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(256, activation="relu"),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(128, activation="relu"),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(10, activation="softmax"),
    ], name="B_dropout"), tf.keras.optimizers.Adam(1e-3)

def config_c():
    return tf.keras.Sequential([
        tf.keras.layers.Input(shape=INPUT_SHAPE),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(512), tf.keras.layers.BatchNormalization(), tf.keras.layers.Activation("relu"),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(256), tf.keras.layers.BatchNormalization(), tf.keras.layers.Activation("relu"),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(128), tf.keras.layers.BatchNormalization(), tf.keras.layers.Activation("relu"),
        tf.keras.layers.Dense(10, activation="softmax"),
    ], name="C_profunda_bn"), tf.keras.optimizers.SGD(1e-2, momentum=0.9)

configs = {"A": config_a, "B": config_b, "C": config_c}

## 4. Entrenamiento

In [ ]:
modelos, historias, resultados = {}, {}, {}

for nombre, builder in configs.items():
    print(f"\n===== Configuración {nombre} =====")
    model, opt = builder()
    model.compile(optimizer=opt,
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])
    model.summary()

    history = model.fit(train_ds, epochs=EPOCHS, validation_data=test_ds, verbose=2)
    _, acc = model.evaluate(test_ds, verbose=0)

    modelos[nombre], historias[nombre], resultados[nombre] = model, history, acc
    print(f"Accuracy en test ({nombre}): {acc:.4f}")
    plot_learning_curve(history, f"Config {nombre}")

## 5. Comparación de accuracy

In [ ]:
for n in configs:
    print(f"Config {n}: test acc = {resultados[n]:.4f} | parámetros = {modelos[n].count_params():,}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for n in configs:
    axes[0].plot(historias[n].history['val_accuracy'], marker="o", label=f"Config {n}")
axes[0].set_title("Validation accuracy por época")
axes[0].set_xlabel("Época"); axes[0].set_ylabel("Accuracy")
axes[0].legend(); axes[0].grid(alpha=0.3)

bars = axes[1].bar(list(resultados), list(resultados.values()), color=["#4C72B0", "#55A868", "#C44E52"])
axes[1].bar_label(bars, fmt="%.4f")
axes[1].set_ylim(min(resultados.values()) - 0.03, 1.0)
axes[1].set_title("Accuracy final en test")
plt.tight_layout()
plt.show()

## 6. Análisis del mejor modelo

In [ ]:
mejor = max(resultados, key=resultados.get)
best_model = modelos[mejor]
print("Mejor configuración:", mejor)

y_true = np.concatenate([y.numpy() for _, y in test_ds])
y_pred = np.argmax(best_model.predict(test_ds, verbose=0), axis=1)

fig, ax = plt.subplots(figsize=(9, 8))
ConfusionMatrixDisplay(confusion_matrix(y_true, y_pred), display_labels=class_names).plot(
    ax=ax, xticks_rotation=45, cmap="Blues", colorbar=False)
ax.set_title(f"Matriz de confusión - Config {mejor}")
plt.tight_layout()
plt.show()

In [ ]:
# Predicciones de ejemplo
imgs, labels = next(iter(test_ds))
preds = best_model.predict(imgs, verbose=0)

plt.figure(figsize=(15, 4))
for i in range(8):
    plt.subplot(1, 8, i + 1)
    plt.imshow(imgs[i].numpy().squeeze(), cmap="gray")
    p = np.argmax(preds[i])
    color = "green" if p == labels[i] else "red"
    plt.title(f"Real: {class_names[labels[i]]}\nPred: {class_names[p]}\n({preds[i][p]:.2f})", color=color, fontsize=9)
    plt.axis("off")
plt.tight_layout()
plt.show()

## 7. Visualización 2D de lo que aprendió la red

Tomamos la salida de la última capa oculta y la proyectamos a 2D con PCA. Si la red separa bien las clases, se forman grupos por color.

In [ ]:
# Modelo que devuelve la penúltima capa (embedding)
embedder = tf.keras.Model(inputs=best_model.inputs, outputs=best_model.layers[-2].output)

N = 3000
x_sub = np.concatenate([x.numpy() for x, _ in test_ds])[:N]
y_sub = y_true[:N]

emb = embedder.predict(x_sub, verbose=0)
emb_2d = PCA(n_components=2).fit_transform(emb)
raw_2d = PCA(n_components=2).fit_transform(x_sub.reshape(N, -1))   # pixeles crudos (solo Flatten)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for ax, data, titulo in [(axes[0], raw_2d, "PCA de pixeles crudos (Flatten)"),
                         (axes[1], emb_2d, f"PCA de la capa oculta - Config {mejor}")]:
    sc = ax.scatter(data[:, 0], data[:, 1], c=y_sub, cmap="tab10", s=6, alpha=0.7)
    ax.set_title(titulo)
handles = [plt.Line2D([], [], marker="o", ls="", color=plt.cm.tab10(i / 9), label=class_names[i]) for i in range(10)]
fig.legend(handles=handles, loc="center right")
plt.tight_layout(rect=[0, 0, 0.88, 1])
plt.show()

## 8. Conclusiones

*(Llenar después de correr el notebook.)*

- Config A: ...
- Config B: ...
- Config C: ...
- Clases que más se confunden (ver matriz): normalmente **Camisa vs Camiseta/Suéter/Abrigo**, porque al aplanar la imagen se pierde la estructura espacial.
- Mejor configuración: ...